[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t01_unit_circle.ipynb)

# The word "sine" is a mistranslation of "bowstring": the unit circle

In 499 CE, aged 23, the Indian astronomer Aryabhata called it *ardha-jya*, "half a bowstring". Arabic scholars wrote
the sound as *jb*. Later readers took it as *jaib*, a fold, and Latin translators in the 1100s made that *sinus*.
So "sine" means a fold, by accident.

In this notebook, all of it comes from one picture: **an arrow of length one, turning around a centre**.

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · The turning arrow and its two projections

Picture an arrow of length one, fixed at the centre. It starts pointing right and turns. Drop a line from its tip
straight down to the horizontal axis. How far across that is from the centre is the arrow's **horizontal
projection**. How far up the tip is, is its **vertical projection**.

The horizontal projection is called the **cosine** of the angle. The vertical projection is the **sine**. That's
the whole definition. Turning anticlockwise is a positive angle. Turning clockwise is a negative angle.

In [ ]:
def turning_arrow(angle_deg=40.0):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    circle_axes(ax)
    turn_arc(ax, a)
    ax.plot([0, c], [0, 0], color=TEAL, lw=7, alpha=0.8, solid_capstyle="butt", label=f"cos = {c:+.3f}  (across)")
    ax.plot([0, 0], [0, s], color=SAGE, lw=7, alpha=0.8, solid_capstyle="butt", label=f"sin = {s:+.3f}  (up)")
    ax.plot([c, c], [0, s], color=TEAL, lw=1, ls=":"); ax.plot([0, c], [s, s], color=SAGE, lw=1, ls=":")
    arrow(ax, a)
    ax.legend(loc="lower left")
    turn = "anticlockwise" if angle_deg >= 0 else "clockwise"
    ax.set_title(f"{angle_deg:.0f}°, turned {turn}")
    plt.show()

interact(turning_arrow, angle_deg=FloatSlider(value=40, min=-360, max=360, step=5, description="angle (°)"));

**Notice.** Drag the angle to 90°. The horizontal projection disappears and the vertical one is the whole arrow:
sin 90° = 1, cos 90° = 0. Now drag it to −60°. The arrow turns the other way, and the vertical projection points
down.

## 2 · Signs in the four quarters

The two projections have directions. Across to the right is positive; to the left, negative. Up is positive; down,
negative. So the quarter of the circle the arrow is in decides the signs.

In [ ]:
def quarters(angle_deg=150.0):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    circle_axes(ax)
    q = int((angle_deg % 360) // 90)
    for k, (x, y, txt) in enumerate(((0.62, 0.62, "sin +\ncos +"), (-0.62, 0.62, "sin +\ncos −"),
                                     (-0.62, -0.62, "sin −\ncos −"), (0.62, -0.62, "sin −\ncos +"))):
        ax.text(x, y, txt, ha="center", va="center", fontsize=15, color=CORAL if k == q else DIM)
    arrow(ax, a)
    ax.plot([0, c], [0, 0], color=TEAL, lw=6, alpha=0.7); ax.plot([0, 0], [0, s], color=SAGE, lw=6, alpha=0.7)
    ax.set_title(f"{angle_deg:.0f}°:  sin {'+' if s >= 0 else '−'},  cos {'+' if c >= 0 else '−'}")
    plt.show()

interact(quarters, angle_deg=FloatSlider(value=150, min=0, max=360, step=5, description="angle (°)"));

**Notice.** Turn past 90°. The horizontal projection crosses to the left, so the cosine turns negative, while the sine stays
positive. Every quarter has its own pair of signs.

In [ ]:
print(f"{'angle':>7} {'sin':>8} {'cos':>8}   quarter")
for d in (30, 150, 210, 330, -30):
    print(f"{d:6d}° {math.sin(math.radians(d)):8.3f} {math.cos(math.radians(d)):8.3f}   {int((d % 360) // 90) + 1}")

## 3 · The arrow never changes length: sin² x + cos² x = 1

The two projections and the arrow make a right triangle: across, up, and the arrow as the long side. Pythagoras says the squares of the short sides add up to the square of the long side. The long side is
always one.

So for every angle, **sin squared x plus cos squared x equals one**. The bars below show it: however you turn, the
two pieces always fill exactly one.

In [ ]:
def never_changes(angle_deg=40.0):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [1, 1.25]})
    circle_axes(ax)
    ax.fill([0, c, c], [0, 0, s], color=CORAL, alpha=0.18)
    ax.plot([0, c], [0, 0], color=TEAL, lw=5); ax.plot([c, c], [0, s], color=SAGE, lw=5)
    arrow(ax, a, label="length 1")
    ax.set_title(f"{angle_deg:.0f}°")
    bx.barh([0], [c * c], color=TEAL, height=0.5, label=f"cos² = {c * c:.3f}")
    bx.barh([0], [s * s], left=[c * c], color=SAGE, height=0.5, label=f"sin² = {s * s:.3f}")
    bx.axvline(1, color=CORAL, lw=2)
    bx.set_xlim(0, 1.15); bx.set_ylim(-0.9, 0.9); bx.set_yticks([]); bx.grid(False)
    bx.set_title(f"cos² + sin² = {c * c + s * s:.12f}")
    bx.legend(loc="upper left")
    plt.tight_layout(); plt.show()

interact(never_changes, angle_deg=FloatSlider(value=40, min=-360, max=360, step=5, description="angle (°)"));

**Notice.** The coral line at one never moves. As one bar grows, the other shrinks by exactly the same amount.

This also tells you the **range**: a projection can never be longer than the arrow. So sine and cosine always stay
between −1 and 1. And the **domain** is every real number, because you can turn by any angle at all.

## 4 · A Ferris wheel is a turning arrow

The original Ferris wheel opened in Chicago in 1893. It had 36 cars, and its top stood 264 feet above the ground. A
seat on a turning wheel is the tip of a turning arrow, so its height is a vertical projection.

Below, the axle is 139 feet up and the wheel's radius is 125 feet. So the height is about
**h = 139 − 125 cos(2π t ÷ T)** feet, where T is the time for one turn. This is an illustrative model built from
those two sizes, not a measurement of the real ride.

In [ ]:
def ferris(minutes=5.0, T=10.0):
    a = 2 * math.pi * minutes / T - math.pi / 2          # start at the bottom
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 5), gridspec_kw={"width_ratios": [1, 1.5]})
    t = np.linspace(0, 2 * np.pi, 300)
    ax.plot(125 * np.cos(t), 139 + 125 * np.sin(t), color=DIM, lw=1.6)
    for k in range(36):
        b = 2 * math.pi * k / 36
        ax.plot([0, 125 * math.cos(b)], [139, 139 + 125 * math.sin(b)], color="#3b4c47", lw=0.8)
    arrow(ax, a, r=125, start=(0, 139))
    h = 139 + 125 * math.sin(a)
    ax.axhline(0, color=DIM); ax.set_xlim(-160, 160); ax.set_ylim(-10, 290); ax.set_aspect("equal")
    ax.set_title(f"seat at {h:.0f} ft"); ax.set_xlabel("feet")
    ts = np.linspace(0, 2 * T, 400)
    bx.plot(ts, 139 - 125 * np.cos(2 * np.pi * ts / T), color=SAGE)
    bx.plot([minutes], [h], "o", color=CORAL, ms=10)
    bx.set_xlabel("minutes"); bx.set_ylabel("height (ft)"); bx.set_ylim(0, 290)
    bx.set_title("height = 139 − 125 cos(2πt ÷ T)")
    plt.tight_layout(); plt.show()

interact(ferris, minutes=FloatSlider(value=5, min=0, max=20, step=0.25, description="time (min)"),
         T=FloatSlider(value=10, min=4, max=20, step=1, description="T (min)"));

**Notice.** The height graph is a wave. It is the seat's vertical projection, drawn over time.

## 5 · Aryabhata's table, in one verse

Aryabhata gave his sines as 24 differences, one for every 3°45′, on a circle of radius 3438. They fit in a single
verse, so a student could learn the whole table by heart. Add them up one by one and you get his sines. Add all 24
and you get the radius.

In [ ]:
DIFFS = [225, 224, 222, 219, 215, 210, 205, 199, 191, 183, 174, 164,
         154, 143, 131, 119, 106, 93, 79, 65, 51, 37, 22, 7]
running = np.cumsum(DIFFS)
angles = 3.75 * np.arange(1, 25)
print("sum of the 24 differences:", running[-1])
assert running[-1] == 3438
print("the circle's circumference in arc-minutes ÷ 2π:", round(21600 / (2 * math.pi), 1))
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(angles, 3438 * np.sin(np.radians(angles)), color=TEAL, lw=6, alpha=0.5, label="3438 × sin (today)")
ax.plot(angles, running, "o", color=CORAL, label="Aryabhata's running sums (499 CE)")
ax.set_xlabel("angle (degrees)"); ax.set_ylabel("half-chord"); ax.legend(loc="lower right")
ax.set_title("Aryabhata's table against modern sines")
plt.show()
print(f"largest gap: {np.max(np.abs(running - 3438 * np.sin(np.radians(angles)))):.1f} (in units of a radius of 3438)")

**Notice.** 3438 is almost exactly 21600 ÷ 2π. A circle has 21600 arc-minutes, so Aryabhata measured his radius in the
same units as the arc. That is the idea behind the radian, more than a thousand years early.

## 6 · The identity gallery

Each identity of this part of the course, three ways: as a fact about the turning arrow, as two graphs that land
on top of each other, and as a check at random angles.

**sin² x + cos² x = 1.** The arrow never changes length.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 600)
fig, ax = plt.subplots(figsize=(9, 3.4))
both_sides(ax, x, np.sin(x) ** 2 + np.cos(x) ** 2, np.ones_like(x), "sin² x + cos² x  and  1", ylim=(-0.2, 1.4))
ax.legend(loc="lower right"); plt.show()
check_identity(lambda x: np.sin(x) ** 2 + np.cos(x) ** 2, lambda x: np.ones_like(x));

**sin(−x) = −sin x** and **cos(−x) = cos x.** Turning the other way is a mirror image in the horizontal axis: the
vertical projection flips, and the horizontal one stays put.

In [ ]:
def mirror(angle_deg=50.0):
    a = math.radians(angle_deg)
    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    circle_axes(ax)
    arrow(ax, a, color=TEAL, label=f"x = {angle_deg:.0f}°")
    arrow(ax, -a, color=CORAL, label=f"−x")
    ax.plot([math.cos(a)] * 2, [math.sin(a), -math.sin(a)], color=DIM, ls=":")
    ax.set_title("−x is x, mirrored in the horizontal axis"); plt.show()

interact(mirror, angle_deg=FloatSlider(value=50, min=0, max=180, step=5, description="x (°)"));

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
both_sides(axs[0], x, np.sin(-x), -np.sin(x), "sin(−x)  and  −sin x")
both_sides(axs[1], x, np.cos(-x), np.cos(x), "cos(−x)  and  cos x")
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(lambda x: np.sin(-x), lambda x: -np.sin(x))
check_identity(lambda x: np.cos(-x), lambda x: np.cos(x));

**sin(x + 2π) = sin x** and **cos(x + 2π) = cos x.** One full turn brings the arrow back where it started.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
both_sides(ax, x, np.sin(x + 2 * np.pi), np.sin(x), "sin(x + 2π)  and  sin x"); ax.legend(loc="lower right"); plt.show()
check_identity(lambda x: np.sin(x + 2 * np.pi), lambda x: np.sin(x))
check_identity(lambda x: np.cos(x + 2 * np.pi), lambda x: np.cos(x));

## 7 · On paper: the CBSE way

**Example 1.** If sin x = 3/5 and x is in the second quarter, find cos x.

1. Use the arrow's length: cos² x = 1 − sin² x = 1 − 9/25 = 16/25.
2. So cos x = 4/5 or −4/5.
3. In the second quarter the horizontal projection points left, so cos x is negative.
4. **cos x = −4/5.**

In [ ]:
x1 = math.pi - math.asin(3 / 5)          # the second-quarter angle with sine 3/5
print(f"x = {math.degrees(x1):.2f}°, sin x = {math.sin(x1):.4f}, cos x = {math.cos(x1):.4f}  (−4/5 = {-4 / 5})")

**Example 2.** Find sin(−11π/3).

1. Add whole turns until the angle is easy: −11π/3 + 2 × 2π = π/3.
2. Whole turns don't change the arrow, so sin(−11π/3) = sin(π/3).
3. **sin(−11π/3) = √3/2.**

In [ ]:
print(f"sin(−11π/3) = {math.sin(-11 * math.pi / 3):.6f},  √3/2 = {math.sqrt(3) / 2:.6f}")

**Try it.**
1. cos x = −1/2 and x is in the third quarter. Find sin x.
2. Find sin 765°.

<details><summary>Answers</summary>

1. sin² x = 1 − 1/4 = 3/4. In the third quarter the vertical projection points down, so sin x = −√3/2.
2. 765° = 2 × 360° + 45°, so sin 765° = sin 45° = √2/2.

</details>

## Check yourself
1. At 200°, what are the signs of sin and cos?
2. Can sin x ever equal 1.2?
3. What is sin² 37° + cos² 37°?

<details><summary>Answers</summary>

1. Both negative: 200° is in the third quarter, so the arrow points left and down.
2. No. A projection can't be longer than the arrow, so sin x is always between −1 and 1.
3. Exactly 1, as for every angle.

</details>

## Sources
MacTutor History of Mathematics (St Andrews): "Trigonometric functions"; "Aryabhata I"; Miller, "Earliest uses of
some words of mathematics" (s). D. Joyce, Clark University, notes on Aryabhata's sine table. Chicago History Museum,
"The Ferris wheel" (36 cars, 264 ft). CBSE Mathematics syllabus 2026–27; NCERT class 11, chapter 3. All retrieved
2026-10-21.